# Modul 19: Sequenzmodelle, Autoencoder und Generierung

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Sequenzmodelle, Autoencoder und Generierung  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Keras-Anwendung mit zeitlichen Daten und Rekonstruktion  
    **Orientierungszeit:** etwa 170 bis 240 Minuten

    ## Überblick

    Sie erstellen zeitlich korrekte Sequenzfenster, vergleichen naive Modelle mit Conv1D, SimpleRNN und GRU und analysieren zeitabhängige Fehler. Danach trainieren Sie dichte Autoencoder für Rekonstruktion, Denoising, Anomaliehinweise und latente Interpolation.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_19A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_19B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Sequenzen in Batch-, Zeit- und Merkmalsachsen strukturieren.
- Zeitliche Splits ohne Zukunftsleckage erstellen und naive Baselines berechnen.
- Conv1D, SimpleRNN und GRU unter gleichen Bedingungen vergleichen.
- Maskierte beziehungsweise gepaddete Sequenzen korrekt behandeln.
- Fehler zeitlich und nach Betriebsabschnitten analysieren.
- Encoder, Decoder, latente Darstellung und Rekonstruktionsverlust praktisch umsetzen.
- Denoising, Anomalieerkennung und latente Interpolation vorsichtig interpretieren.

    ## Bewertete Fähigkeiten

    - Sequenzfenster, zeitliche Splits und naive Baselines
- Conv1D, SimpleRNN und GRU in Keras
- Masking, Padding und zeitliche Fehleranalyse
- dichte Autoencoder, latente Codes und Rekonstruktionsverlust
- Denoising, Anomalieschwellen und latente Interpolation

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.datasets import load_digits
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

# Eine lokale synthetische Messreihe kombiniert mehrere Frequenzen,
# einen langsamen Trend und reproduzierbares Rauschen.
sequence_rng_19 = np.random.default_rng(RANDOM_SEED)
time_index_19 = np.arange(0, 1800, dtype=np.float32)
raw_signal_19 = (
    np.sin(time_index_19 / 18.0)
    + 0.35 * np.sin(time_index_19 / 5.5)
    + 0.0007 * time_index_19
    + sequence_rng_19.normal(0.0, 0.08, size=len(time_index_19))
).astype("float32")

digits_19 = load_digits()
digit_images_19 = (digits_19.images.astype("float32") / 16.0)
digit_vectors_19 = digit_images_19.reshape(len(digit_images_19), -1)
digit_labels_19 = digits_19.target.astype("int64")
AE_train_valid_19, AE_test_19, AE_y_train_valid_19, AE_y_test_19 = train_test_split(
    digit_vectors_19,
    digit_labels_19,
    test_size=0.20,
    stratify=digit_labels_19,
    random_state=RANDOM_SEED,
)
AE_train_19, AE_valid_19, AE_y_train_19, AE_y_valid_19 = train_test_split(
    AE_train_valid_19,
    AE_y_train_valid_19,
    test_size=0.25,
    stratify=AE_y_train_valid_19,
    random_state=RANDOM_SEED,
)

print("Rohsequenz:", raw_signal_19.shape)
print("Autoencoder Train/Valid/Test:", AE_train_19.shape, AE_valid_19.shape, AE_test_19.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Sequenzfenster, zeitliche Splits und Baselines

    Bereiten Sie die synthetische Messreihe als Vorhersageproblem vor.

1. Schreiben Sie eine Funktion, die aus einer eindimensionalen Reihe Fenster der Länge 24 und jeweils den direkt folgenden Zielwert erzeugt.
2. Bringen Sie die Merkmale in die Form `(Beispiele, Zeitschritte, Merkmale)`.
3. Teilen Sie chronologisch in 60 Prozent Training, 20 Prozent Validierung und 20 Prozent Test. Verwenden Sie kein zufälliges Shuffling vor dem Split.
4. Berechnen Sie eine naive Persistenzbaseline, die den letzten Fensterwert vorhersagt.
5. Trainieren Sie zusätzlich Ridge Regression auf den abgeflachten Trainingsfenstern.
6. Vergleichen Sie Validierungs-MAE und Test-MAE beider Baselines und visualisieren Sie einen Ausschnitt der Testvorhersagen.

> **Hinweis:** Das Ziel eines Fensters beginnt genau an der Position direkt hinter dem letzten Eingabewert.

In [ ]:
window_length_19 = 24

# Speichern Sie die resultierenden Partitionen als X_seq_train_19,
# X_seq_valid_19, X_seq_test_19 und entsprechende y-Variablen.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Das Ziel eines Fensters beginnt genau an der Position direkt hinter dem letzten Eingabewert.

## Aufgabe 2: Conv1D, SimpleRNN und GRU fair vergleichen

    Trainieren Sie drei kleine Sequenzmodelle unter gleichen Bedingungen.

1. Erstellen Sie je ein Modell mit `Conv1D`, `SimpleRNN` und `GRU`.
2. Halten Sie Eingabeform, Ausgabeschicht, Optimierer, Loss, maximale Epochen und Batchgröße gleichartig.
3. Verwenden Sie MAE als Loss und Metrik sowie Early Stopping auf `val_loss`.
4. Speichern Sie Trainingsdauer, beste Validierungs-MAE, Test-MAE und Parameterzahl.
5. Wählen Sie das beste Modell ausschließlich anhand der Validierung und speichern Sie es als `best_sequence_model_19`.
6. Vergleichen Sie das ausgewählte Modell mit Persistenz und Ridge auf dem Testset.

> **Hinweis:** Erstellen Sie für jedes Modell eine neue Instanz und setzen Sie den Seed vor dem Aufbau.

In [ ]:
sequence_epochs_19 = 4 if FAST_MODE else 25
sequence_batch_size_19 = 32

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Erstellen Sie für jedes Modell eine neue Instanz und setzen Sie den Seed vor dem Aufbau.

## Aufgabe 3: Maskierte Sequenzen und zeitliche Fehleranalyse

    Untersuchen Sie variable Sequenzlängen und die Fehlerentwicklung über die Zeit.

1. Erstellen Sie mehrere Sequenzen unterschiedlicher Länge und padden Sie sie rechts mit dem Wert `-999.0`.
2. Bauen Sie ein kleines Modell aus `Masking`, `GRU` und Dense-Ausgabe.
3. Zeigen Sie mit zwei identischen gültigen Sequenzen und unterschiedlich viel Padding, dass maskierte Zusatzwerte die Ausgabe praktisch nicht verändern.
4. Berechnen Sie für das in Aufgabe 2 ausgewählte Sequenzmodell absolute Testfehler.
5. Fassen Sie die Fehler für vier aufeinanderfolgende zeitliche Testabschnitte zusammen.
6. Visualisieren Sie wahre Werte, Vorhersagen und absoluten Fehler über einen längeren Testausschnitt.

> **Hinweis:** Padding muss immer denselben speziellen Wert verwenden wie die Masking-Schicht.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Padding muss immer denselben speziellen Wert verwenden wie die Masking-Schicht.

## Aufgabe 4: Einen dichten Autoencoder trainieren und den latenten Raum untersuchen

    Trainieren Sie einen Autoencoder auf den kleinen Ziffernbildern.

1. Erstellen Sie einen Encoder `64 -> 32 -> 12` und einen Decoder `12 -> 32 -> 64`.
2. Verwenden Sie Sigmoid in der Rekonstruktionsausgabe und MSE als Loss.
3. Trainieren Sie mit Eingabe gleich Ziel und Early Stopping.
4. Berechnen Sie den Rekonstruktions-MSE auf Training, Validierung und Test.
5. Visualisieren Sie sechs Originale und Rekonstruktionen.
6. Erzeugen Sie latente Codes für das Testset und stellen Sie die ersten beiden latenten Dimensionen nach Ziffernklasse dar. Interpretieren Sie diese Projektion vorsichtig.

> **Hinweis:** Eingabe und Ziel sind beim klassischen Autoencoder identisch.

In [ ]:
autoencoder_epochs_19 = 6 if FAST_MODE else 35
latent_dimension_19 = 12

# Speichern Sie die Modelle als encoder_19, decoder_19 und autoencoder_19.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Eingabe und Ziel sind beim klassischen Autoencoder identisch.

## Aufgabe 5: Integration: Denoising, Anomaliehinweise und latente Interpolation

    Erweitern Sie den Autoencoder-Workflow.

1. Fügen Sie den Trainingsbildern reproduzierbares Gaußrauschen hinzu und begrenzen Sie Werte auf `[0, 1]`.
2. Trainieren Sie einen neuen Autoencoder, der verrauschte Eingaben auf saubere Ziele abbildet.
3. Visualisieren Sie verrauschte Eingaben, saubere Bilder und Denoising-Rekonstruktionen.
4. Bestimmen Sie eine Anomalieschwelle als 95. Perzentil der Rekonstruktionsfehler auf **sauberen Validierungsdaten**.
5. Erzeugen Sie künstliche Anomalien durch zufälliges Mischen der Pixel einiger Testbilder und prüfen Sie, welcher Anteil oberhalb der Schwelle liegt.
6. Interpolieren Sie im latenten Raum zwischen zwei Testziffern und decodieren Sie fünf Zwischenpunkte.
7. Erläutern Sie Grenzen einer solchen Anomalieentscheidung und einer Interpretation interpolierter Bilder als echte Datengenerierung.

> **Hinweis:** Bestimmen Sie die Schwelle ohne Testanomalien, sonst wird die Bewertung verzerrt.

In [ ]:
denoising_epochs_19 = 6 if FAST_MODE else 35

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Bestimmen Sie die Schwelle ohne Testanomalien, sonst wird die Bewertung verzerrt.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.